# Search harness walkthrough (ABR)

For developers who are new to search or this repo: you will **see the SDK imports and parameters** in the cells below—only CSV path discovery is a few lines of `pathlib`.

- **Typesense** holds the ABR documents.
- **`SearchClient`** + **`indexes.create(...)`** register an index backed by **`TypesenseAdapter`**.
- **`QueryAnalyzer`** wraps a model provider (**`AbrNotebookProvider`** here—a small demo, no live LLM).

**Before you run:** `uv sync`, **Docker** for Typesense, `data/abr_merged.csv`, kernel = project venv, **cwd = repository root** so `import examples.…` works.

## 1. Imports

| What | Why |
|------|-----|
| `SearchClient` | Entry point; owns traces and indexes. |
| `QueryAnalyzer` | Runs classification/extraction before each search. |
| `InteractionMode` | Default behavior for the index (here: AITL). |
| `TypesenseAdapter`, `create_collection_if_missing` | Backend adapter + schema sync with Typesense. |
| `examples.abr_typesense_helpers` | ABR schema, demo provider, Typesense client, CSV import. |

In [2]:
from __future__ import annotations

from pathlib import Path
from pprint import pprint

from search_service import InteractionMode, QueryAnalyzer, SearchClient
from search_service.adapters import TypesenseAdapter, create_collection_if_missing

from examples.abr_typesense_helpers import (
    AbrNotebookProvider,
    SEARCHABLE_FIELDS,
    build_abr_typesense_config,
    build_typesense_client,
    ensure_typesense_container,
    get_collection_document_count,
    import_abr_documents_to_typesense,
    preview_abr_documents,
    recreate_collection,
    wait_for_typesense,
)

## 2. Paths and a sample row

`preview_abr_documents` reads only a few valid rows (bounded scan)—it does not load the whole CSV.

In [3]:
def project_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("Run from the repo root (pyproject.toml not found).")


ROOT = project_root()
CSV_PATH = ROOT / "data" / "abr_merged.csv"
assert CSV_PATH.exists(), f"Missing {CSV_PATH}"

sample = preview_abr_documents(CSV_PATH, limit=1)
pprint(sample[0] if sample else "(no rows parsed)")

{'abn': '11000000948',
 'abn_status': 'ACT',
 'all_other_entity_names': 'QBE INSURANCE (INTERNATIONAL) LIMITED',
 'business_names': '',
 'dgr_status': '',
 'entity_name': 'QBE INSURANCE (INTERNATIONAL) LTD',
 'entity_name_type': 'MN',
 'entity_type_ind': 'PUB',
 'entity_type_text': 'Australian Public Company',
 'gst_status': 'ACT',
 'id': '11000000948',
 'legal_full_name': '',
 'main_name': 'QBE INSURANCE (INTERNATIONAL) LTD',
 'other_names': '',
 'postcode': '2000',
 'record_last_updated_date': '20180216',
 'replaced': 'N',
 'source_file': 'public_split_1_10/20260325_Public01.xml',
 'state': 'NSW',
 'trading_names': 'QBE INSURANCE (INTERNATIONAL) LIMITED'}


## 3. Typesense + `IndexConfig` + `SearchClient`

Steps match the idea of `build_abr_typesense_index` in `abr_typesense_helpers.py`, but **spelled out here**:

1. Start Typesense (Docker) and wait until it answers `/health`.
2. Build a Typesense Python client (`host`, `port`, `api_key`, `protocol`).
3. **`TypesenseAdapter(ts_client, collection_name, SEARCHABLE_FIELDS)`** — tells the SDK how to query Typesense.
4. **`build_abr_typesense_config(...)`** — builds **`IndexConfig`** (schema, filters, policy, default interaction mode).
5. **`create_collection_if_missing`** — creates the collection from that config if needed.
6. **`import_abr_documents_to_typesense`** — streams the CSV into Typesense (honours `limit` / `batch_size`).
7. **`SearchClient()`** + **`client.indexes.create(config, analyzer=QueryAnalyzer(AbrNotebookProvider()))`** — registers the index the notebook will call **`index.search(...)`** on.

Adjust **`IMPORT_LIMIT`**, **`REINDEX`**, and Typesense connection fields in the next cell.

In [4]:
# --- Typesense connection ---
TYPESENSE_HOST = "localhost"
TYPESENSE_PORT = 8108
TYPESENSE_API_KEY = "search-service-dev"
TYPESENSE_PROTOCOL = "http"

# --- Index / import ---
INDEX_NAME = "abr_entities_walkthrough"
IMPORT_LIMIT = 2_500
IMPORT_BATCH_SIZE = 5_000
REINDEX = False  # True: delete collection then re-import

ensure_typesense_container()
wait_for_typesense(
    host=TYPESENSE_HOST,
    port=TYPESENSE_PORT,
    protocol=TYPESENSE_PROTOCOL,
)

ts_client = build_typesense_client(
    host=TYPESENSE_HOST,
    port=TYPESENSE_PORT,
    api_key=TYPESENSE_API_KEY,
    protocol=TYPESENSE_PROTOCOL,
)

adapter = TypesenseAdapter(ts_client, INDEX_NAME, SEARCHABLE_FIELDS)

config = build_abr_typesense_config(
    adapter,
    interaction_mode=InteractionMode.aitl,
    index_name=INDEX_NAME,
)

import_summary = None
if REINDEX:
    recreate_collection(ts_client, INDEX_NAME)

create_collection_if_missing(ts_client, config)

if REINDEX or get_collection_document_count(ts_client, INDEX_NAME) == 0:
    import_summary = import_abr_documents_to_typesense(
        ts_client,
        INDEX_NAME,
        CSV_PATH,
        limit=IMPORT_LIMIT,
        batch_size=IMPORT_BATCH_SIZE,
    )

client = SearchClient()
analyzer = QueryAnalyzer(AbrNotebookProvider())
index = client.indexes.create(config, analyzer=analyzer)

print("documents in Typesense:", get_collection_document_count(ts_client, INDEX_NAME))
pprint(import_summary)

documents in Typesense: 2500
None


## 4. Search in plain language

The analyzer turns phrases like “NSW” or “active” into **filters** (see `AbrNotebookProvider.extract_entities` in `abr_typesense_helpers.py`). Below we read **`SearchResultEnvelope`** fields directly.

In [6]:
r = index.search("QBE NSW active")
print("status:", r.status.value, "| trace_id:", r.trace_id)
print("message:", r.message)
if r.query_analysis is not None:
    pprint(r.query_analysis.model_dump())
for i, item in enumerate(r.results[:5], 1):
    print(i, item.title, "|", {k: item.metadata.get(k) for k in ("abn", "state", "abn_status") if item.metadata.get(k)})

status: completed | trace_id: 7880c73464af44588fd3f6af8e495244
message: Search completed with 2 result(s) (1 iteration(s), 2 branch(es))
{'ambiguity': <AmbiguityLevel.none: 'none'>,
 'extracted_entities': [],
 'filters': {'abn_status': 'ACT', 'state': 'NSW'},
 'missing_fields': [],
 'possible_resource_types': [],
 'primary_subject': 'QBE NSW active',
 'query_type': 'entity_lookup',
 'raw_query': 'QBE NSW active',
 'target_resource_type': 'company'}
1 QBE INSURANCE (INTERNATIONAL) LTD | {'abn': '11000000948', 'state': 'NSW', 'abn_status': 'ACT'}


## 5. Ambiguity → `needs_input` → `continue_search`

Same **`trace_id`** ties the follow-up to the original request.

In [9]:
r = index.search("ambiguous Telstra")
print("status:", r.status.value)
print("follow_up:", r.follow_up.model_dump() if r.follow_up else None)

if r.follow_up is not None:
    r2 = index.continue_search(r.trace_id, {"state": "NSW"})
    print("after continue_search — status:", r2.status.value)
    for i, item in enumerate(r2.results[:5], 1):
        print(i, item.title)

status: needs_input
follow_up: {'reason': 'ambiguous_entity', 'message': 'The search could not proceed with enough confidence. Please narrow your query or provide additional detail.', 'input_schema': {'type': 'object', 'properties': {'state': {'type': 'string', 'enum': ['NSW', 'VIC', 'QLD', 'SA', 'WA', 'TAS', 'NT', 'ACT']}, 'abn_status': {'type': 'string', 'enum': ['ACT', 'CAN']}, 'entity_type_text': {'type': 'string', 'enum': ['Australian Private Company', 'Australian Public Company', 'Individual/Sole Trader', 'Individual/Sole Trader', 'Family Partnership', 'Other Partnership', 'Other Partnership', 'ATO Regulated Self-Managed Superannuation Fund', 'ATO Regulated Self-Managed Superannuation Fund', 'Discretionary Trading Trust', 'Fixed Unit Trust']}, 'entity_type_ind': {'type': 'string'}, 'entity_name_type': {'type': 'string'}, 'postcode': {'type': 'string'}, 'gst_status': {'type': 'string', 'enum': ['ACT', 'CAN', 'NON']}, 'dgr_status': {'type': 'string', 'enum': ['ACT']}, 'replaced': {

## 6. Pass `filters=` yourself

Filters must match fields declared on the index (`filterable_fields` in `build_abr_typesense_config`).

In [11]:
r = index.search(
    "business",
    filters={"state": "VIC", "abn_status": "ACT"},
)
print("status:", r.status.value, "hits:", len(r.results))
for item in r.results[:5]:
    print(item.title)

status: completed hits: 11
TAREE COMPUTER BUSINESS PTY LTD
KARENMARK BUSINESS SERVICES PTY. LTD.
BUSINESS AND FREEHOLD VALUERS PTY. LTD.
DYNAMIC BUSINESS CENTRES PTY LTD
BUSINESS BARTER EXCHANGE (AUSTRALIA) PTY LIMITED


## 7. Trace: `client.tracer.get(trace_id)`

The **`SearchClient`** you created owns the in-memory tracer for this session.

In [19]:
r = index.search("QLD")

In [20]:
r.model_dump()

{'status': <SearchStatus.completed: 'completed'>,
 'original_query': 'QLD',
 'interaction_mode': <InteractionMode.aitl: 'aitl'>,
 'query_analysis': {'raw_query': 'QLD',
  'query_type': 'entity_lookup',
  'primary_subject': 'QLD',
  'target_resource_type': 'company',
  'possible_resource_types': [],
  'filters': {'state': 'QLD'},
  'ambiguity': <AmbiguityLevel.none: 'none'>,
  'missing_fields': [],
  'extracted_entities': []},
 'results': [{'id': '11003302903',
   'title': 'SDC (QLD) PTY LTD',
   'snippet': None,
   'score': None,
   'source': 'abr_entities_walkthrough',
   'matched_fields': [],
   'metadata': {'abn': '11003302903',
    'entity_name': 'SDC (QLD) PTY LTD',
    'main_name': 'SDC (QLD) PTY LTD',
    'entity_type_text': 'Australian Private Company',
    'abn_status': 'CAN',
    'state': 'NSW',
    'postcode': '2077',
    'gst_status': 'CAN',
    'trading_names': '',
    'business_names': ''}}],
 'branches': [{'kind': <BranchKind.original_query: 'original_query'>,
   'query'

In [ ]:
trace = client.tracer.get(r.trace_id)

In [ ]:
r = index.search("sole trader QLD")

assert trace is not None
print("iterations_used:", trace.iterations_used, "branches_used:", trace.branches_used)
for step in trace.steps:
    payload = step.payload
    if isinstance(payload, dict) and len(str(payload)) > 220:
        payload = f"<dict len={len(payload)}>"
    print(step.step_type.value, "→", payload)

---

**Next:** `examples/sdk_playground.ipynb` for more scenarios; **`examples/abr_typesense_helpers.py`** for `build_abr_typesense_index` (one-call equivalent of section 3) and CSV helpers. Set **`REINDEX = True`** once if you need to drop and reload the collection.